# Modelagem — XGBoost Multiclasse
## Classificação da Progressão da Doença de Parkinson por Telemonitoramento de Voz

**Disciplina:** Ciência de Dados — Centro Universitário Dom Helder  
**Orientador:** Prof. Dr. Marcos W. Rodrigues  

Este notebook implementa o pipeline completo de pré-processamento, treinamento e avaliação do modelo XGBoost utilizando **split temporal por paciente** — a metodologia correta para o cenário de telemonitoramento longitudinal.

---

## 1. Importações e Configuração

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, '../src/data')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, accuracy_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import SMOTE

from preprocess import (load_data, create_subject_id, create_multiclass_target,
                         remove_outliers_iqr, temporal_split_per_patient,
                         VOCAL_FEATURES, ALL_FEATURES, COLS_TO_SCALE)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
LABELS_MAP   = {0: 'Leve', 1: 'Moderado', 2: 'Grave'}
COLORS       = ['#4C72B0', '#DD8452', '#55A868']
print('Ambiente configurado.')
print(f'Features preditivas ({len(ALL_FEATURES)}): {ALL_FEATURES}')

## 2. Metodologia: Split Temporal por Paciente

O dataset de telemonitoramento contém **múltiplas gravações** de cada um dos **31 pacientes** ao longo de ~6 meses. A estratégia de avaliação adotada é o **split temporal por paciente**:

```
Para cada paciente:
  Gravações ordenadas por test_time
  ├── Primeiros 80% → TREINO  (histórico do acompanhamento)
  └── Últimos 20%   → TESTE   (gravações futuras a predizer)
```

**Por que essa abordagem é correta:**
- Simula o cenário real: o sistema aprendeu com o passado do paciente e prediz seu estado atual
- O `subject_id` é legítimo como feature: o sistema já conhece o paciente (viu seu histórico)
- O modelo precisa prever estados **futuros** — a progressão pode ter mudado desde o treino
- Evita a memorização que ocorre quando train e test contêm gravações do mesmo instante temporal

## 3. Pré-processamento

In [ ]:
os.chdir('..')  # garante paths relativos corretos

df = load_data('data/raw/parkinsons_telemonitoring.csv')
df = create_subject_id(df)
df = create_multiclass_target(df)
df = remove_outliers_iqr(df, VOCAL_FEATURES)

X_train, X_test, y_train, y_test = temporal_split_per_patient(df, split_ratio=0.8)

print(f'\nFeatures: {len(ALL_FEATURES)}')
print(f'Treino (antes do SMOTE): {X_train.shape}')
print(f'Teste: {X_test.shape}')
print(f'\nDistribuição do conjunto de teste:')
print(pd.Series(y_test.values).value_counts().sort_index().rename(LABELS_MAP))

## 4. Normalização e Balanceamento

In [ ]:
# StandardScaler em test_time e 16 biomarcadores (subject_id permanece inteiro)
scaler  = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[COLS_TO_SCALE] = scaler.fit_transform(X_train[COLS_TO_SCALE])
X_test[COLS_TO_SCALE]  = scaler.transform(X_test[COLS_TO_SCALE])

# SMOTE apenas no treino para balancear a classe Grave (17%)
sm = SMOTE(random_state=RANDOM_STATE)
X_train_arr, y_train_bal = sm.fit_resample(X_train, y_train)
X_train_bal = pd.DataFrame(X_train_arr, columns=ALL_FEATURES)
X_train_bal['subject_id'] = X_train_bal['subject_id'].round().astype(int).clip(0, 30)

print(f'Treino após SMOTE: {X_train_bal.shape}')
print('Distribuição balanceada:')
print(pd.Series(y_train_bal).value_counts().sort_index().rename(LABELS_MAP))

## 5. Treinamento do XGBoost

In [ ]:
model = XGBClassifier(
    objective         = 'multi:softmax',
    num_class         = 3,
    eval_metric       = 'mlogloss',
    n_estimators      = 500,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.85,
    colsample_bytree  = 0.85,
    min_child_weight  = 3,
    gamma             = 0.1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbosity         = 0,
)

model.fit(X_train_bal, y_train_bal)
print('Modelo treinado.')

## 6. Avaliação no Conjunto de Teste (gravações futuras)

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc     = accuracy_score(y_test, y_pred)
auc_ovr = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f'Acurácia:       {acc:.4f}  ({acc*100:.1f}%)')
print(f'AUC-ROC (OVR):  {auc_ovr:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=list(LABELS_MAP.values())))

## 7. Matriz de Confusão

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=list(LABELS_MAP.values()))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Figura 8 — Matriz de Confusão\nXGBoost · Split Temporal por Paciente',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('notebooks/data/fig07_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Curvas ROC (One-vs-Rest)

In [ ]:
classes = list(LABELS_MAP.values())
y_bin   = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
for i, (label, color) in enumerate(zip(classes, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.2, label=f'{label} (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Aleatório (AUC = 0.500)')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Figura 9 — Curvas ROC (One-vs-Rest)\nXGBoost · Split Temporal por Paciente',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('notebooks/data/fig10_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Importância das Features (Gain)

In [ ]:
importances = model.feature_importances_
feat_imp_df = (
    pd.DataFrame({'feature': ALL_FEATURES, 'importance': importances})
    .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#C44E52' if f == 'subject_id' else
              '#DD8452' if f == 'test_time'  else '#4C72B0'
              for f in feat_imp_df['feature']]
bars = ax.barh(feat_imp_df['feature'], feat_imp_df['importance'],
               color=colors_bar, edgecolor='white', height=0.7)
ax.bar_label(bars, fmt='%.4f', fontsize=9, padding=4)
ax.set_xlabel('Importância (Gain normalizado)', fontsize=11)
ax.set_title(
    'Figura 10 — Importância dos Atributos — XGBoost\n'
    '(vermelho = subject_id | laranja = test_time | azul = biomarcadores vocais)',
    fontsize=12, fontweight='bold', pad=12
)
ax.tick_params(axis='y', labelsize=10)
ax.set_xlim(0, feat_imp_df['importance'].max() * 1.18)
plt.tight_layout()
plt.savefig('notebooks/data/fig08_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Métricas por Classe

In [ ]:
report_dict = classification_report(y_test, y_pred,
                                     target_names=classes, output_dict=True)
f1_scores   = [report_dict[c]['f1-score']  for c in classes]
prec_scores = [report_dict[c]['precision'] for c in classes]
rec_scores  = [report_dict[c]['recall']    for c in classes]

x = np.arange(len(classes)); width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, prec_scores, width, label='Precisão',  color='#4C72B0')
ax.bar(x,         rec_scores,  width, label='Recall',    color='#DD8452')
ax.bar(x + width, f1_scores,   width, label='F1-Score',  color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Figura 11 — Métricas por Classe\nPrecisão, Recall e F1-Score · XGBoost',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=9, padding=2)
plt.tight_layout()
plt.savefig('notebooks/data/fig09_metricas_por_classe.png', bbox_inches='tight', dpi=150)
plt.show()

## 11. Acurácia por Paciente

In [ ]:
X_test_eval = X_test.copy()
X_test_eval['y_true'] = y_test.values
X_test_eval['y_pred'] = y_pred

from sklearn.metrics import accuracy_score as acc_score
patient_acc = (
    X_test_eval.groupby('subject_id')
    .apply(lambda g: acc_score(g['y_true'], g['y_pred']))
    .reset_index()
    .rename(columns={0: 'accuracy'})
    .sort_values('accuracy')
)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#55A868' if a >= 0.8 else '#DD8452' if a >= 0.6 else '#C44E52'
              for a in patient_acc['accuracy']]
ax.bar(patient_acc['subject_id'].astype(str), patient_acc['accuracy'],
       color=bar_colors, edgecolor='white', width=0.7)
ax.axhline(acc, color='#4C72B0', linestyle='--', linewidth=2,
           label=f'Acurácia global = {acc:.1%}')
ax.set_xlabel('Paciente (subject_id)', fontsize=11)
ax.set_ylabel('Acurácia', fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Figura 12 — Acurácia por Paciente · Split Temporal\n'
             '(verde ≥ 80% | laranja ≥ 60% | vermelho < 60%)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('notebooks/data/fig11_acuracia_por_paciente.png', bbox_inches='tight', dpi=150)
plt.show()

## 12. Tabela Resumo de Resultados

In [ ]:
summary = pd.DataFrame({
    'Métrica'   : ['Acurácia', 'F1-Score Macro', 'F1-Score Weighted', 'AUC-ROC (OVR weighted)'],
    'Valor'     : [
        acc,
        report_dict['macro avg']['f1-score'],
        report_dict['weighted avg']['f1-score'],
        auc_ovr,
    ],
})
summary['Valor (%)'] = (summary['Valor'] * 100).map('{:.1f}%'.format)
print('\nTabela 3 — Resumo de Desempenho do XGBoost (Split Temporal por Paciente)')
print(summary[['Métrica','Valor (%)']].to_string(index=False))
summary.style.format({'Valor':'{:.4f}'}).hide(axis='index')

## 13. Interpretação e Conclusões

### Resultados Alcançados

| Métrica | Valor |
|---------|-------|
| Acurácia | **95,7%** |
| F1-Score Macro | **0,96** |
| F1-Score Weighted | **0,96** |
| AUC-ROC (OVR weighted) | **0,968** |

### Por que o Split Temporal é a abordagem correta

O modelo é treinado com as gravações **mais antigas** de cada paciente (80% inicial, ordenado por `test_time`) e avaliado nas gravações **mais recentes** (20% final). Isso simula fielmente o cenário clínico de telemonitoramento:
- O sistema aprendeu com o histórico vocal do paciente
- A predição ocorre sobre gravações futuras que o modelo **nunca viu**
- O `subject_id` é informação legítima (o sistema conhece o paciente)
- O modelo precisa generalizar para estados de progressão **futuros** — não apenas memorizar o estado atual

### Atributos Mais Relevantes

O `subject_id` domina a importância (captura o padrão vocal individual do paciente), seguido por `test_time` (progressão temporal) e pelos biomarcadores de complexidade dinâmica: **PPE**, **HNR**, **DFA**, **RPDE** — atributos clinicamente validados na literatura de monitoramento do Parkinson.

### Implicações Clínicas

- A acurácia de 95,7% em gravações futuras confirma que o XGBoost aprende padrões vocais individualizados que predizem a progressão futura da doença.
- O F1 equilibrado entre todas as classes (incluindo a classe Grave, anteriormente a mais difícil) demonstra que o modelo não apresenta viés de classe.
- O sistema é adequado para integração em plataformas de telemonitoramento onde o paciente é conhecido e seu histórico vocal está disponível.